In [41]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
import numpy as np
import torch
import gymnasium as gym
from typing import Callable, cast, List, Tuple, Union

from student_client import create_student_gym_env, StudentGymEnv
from DQN_utils import train_dqn, format_state, train_only_dqn, QNetwork

In [43]:
device = torch.device("cpu" if torch.mps.is_available() else "cpu") 
loaded_network = QNetwork(100, 3, 128, 64).to(device)

checkpoint = torch.load("./checkpoints/q_network_ep_150.pth", weights_only=False, map_location=device)
loaded_network.load_state_dict(checkpoint['model_state_dict'])

optimizer = torch.optim.Adam(loaded_network.parameters(), lr=0.1, )
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

In [ ]:
# env = create_student_gym_env(user_token='14S544O#BEiKzhK')

trained_dqn, rewards, memory = train_dqn(num_episodes=500, batch_size=256, gamma=0.9, learning_rate=1e-4, epsilon_start=0.9, epsilon_end=0.05, device=device)

Error: File 'air_palaiseau_buffer.pkl' not found.


Training DQN:   0%|          | 0/500 [00:00<?, ?it/s]

2026-03-03 14:16:21,468 - student_client.student_gym_env - INFO - StudentGymEnv initialized with episode b1bfeaa5-080f-46e5-91dd-282e165d9a98
2026-03-03 14:16:22,761 - student_client.student_gym_env - INFO - Episode b1bfeaa5-080f-46e5-91dd-282e165d9a98 reset successfully


action: 0
action: 1, tensor([[-51385.7969,  43931.8672,  33809.3320]])


KeyboardInterrupt: 

In [22]:
trained_only_dqn = train_only_dqn(
    num_epochs=50000,
    batch_size=64,
    gamma=0.995,
    learning_rate=1e-5,
    target_update_freq=10,
    device=device,
)

Buffer loaded successfully from air_palaiseau_buffer.pkl
Loaded 2795 normal steps and 47 failures.
Epoch 1/50000 | Loss: 1097805184.00
Epoch 2/50000 | Loss: 963480576.00
Epoch 3/50000 | Loss: 965129728.00
Epoch 4/50000 | Loss: 840462080.00
Epoch 5/50000 | Loss: 969190016.00
Epoch 6/50000 | Loss: 917540160.00
Epoch 7/50000 | Loss: 972906688.00
Epoch 8/50000 | Loss: 790479616.00
Epoch 9/50000 | Loss: 769888000.00
Epoch 10/50000 | Loss: 742851072.00
Epoch 11/50000 | Loss: 708482816.00
Epoch 12/50000 | Loss: 717554816.00
Epoch 13/50000 | Loss: 767728896.00
Epoch 14/50000 | Loss: 708287936.00
Epoch 15/50000 | Loss: 596626880.00
Epoch 16/50000 | Loss: 828056512.00
Epoch 17/50000 | Loss: 804158336.00
Epoch 18/50000 | Loss: 743091776.00
Epoch 19/50000 | Loss: 701606912.00
Epoch 20/50000 | Loss: 743257600.00
Epoch 21/50000 | Loss: 691004672.00
Epoch 22/50000 | Loss: 732466624.00
Epoch 23/50000 | Loss: 701196480.00
Epoch 24/50000 | Loss: 683253632.00
Epoch 25/50000 | Loss: 528287520.00
Epoch 26/

In [ ]:
new_trained_dqn, rewards, memory = train_dqn(num_episodes=500, batch_size=128, gamma=0.9, learning_rate=1e-4, epsilon_start=0.01, device=device, checkpoint_network=trained_only_dqn)

In [ ]:
import random
# Initialize data collection arrays
env = create_student_gym_env(user_token='14S544O#BEiKzhK')
obs, info = env.reset()
observations = [obs]
actions = []
rewards = []
total_timesteps = 0
epsilon = 0.01

for step in range(50):

    if random.random() < epsilon:
        print("random")
        action = env.action_space.sample() # Explore
    else:
        print("exploit")
        state = format_state(observations[-1])
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            q_values = trained_dqn(state_tensor)
        print(q_values)
        action = q_values.argmax(dim=1).item()

    obs_result, reward, terminated, truncated, info = env.step(
        action=action,
    )

    observations.append(obs_result)
    actions.append(action)

    rewards.append(reward)

    total_timesteps = info['step']

    if step % 1 == 0:
        print(f" Step {total_timesteps}: Reward={reward:.2f}, Total={sum(rewards):.2f}")

    if terminated or truncated:
        print(f"🏁 Episode ended at step {total_timesteps} with reward={reward:.2f}")
        break


# Print summary statistics
total_reward = sum(rewards)
print(f"\n Episode Summary:")
print(f"   Total Steps: {len(actions)}")
print(f"   Total Reward: {total_reward:.2f}")
print(f"   Actions Taken: {len([a for a in actions if a == 1])} repairs, {len([a for a in actions if a == 2])} sell")

In [ ]:
from student_client.plotting import plot_observations


plot_observations(
        observations=observations,
        actions=actions,
        title="Simple Policy - Observation Dimensions Over Time"
    )

In [ ]:
from student_client.plotting import plot_rewards

plot_rewards(rewards=rewards)